# Day 75: Build a Reusable Inference Benchmark Harness

**Layer:** Production


## Concept Overview

A benchmark harness runs standardized workloads against any OpenAI-compatible server, producing reproducible TTFT/TPOT/throughput reports.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Benchmark Harness

A harness sends a configurable mix of prompt lengths and measures all key metrics.


In [ ]:
import asyncio, time, json, numpy as np

class InferenceBenchmarkHarness:
    def __init__(self, base_url="http://localhost:8000", model="default"):
        self.base_url = base_url
        self.model = model

    async def send_request(self, prompt_len, output_len, sem):
        async with sem:
            # Simulate request
            ttft_ms = prompt_len * 0.05 + np.random.normal(50, 5)
            tpot_ms = np.random.normal(20, 2)
            total_ms = ttft_ms + output_len * tpot_ms
            await asyncio.sleep(total_ms/1000)
            return {"ttft_ms":ttft_ms,"tpot_ms":tpot_ms,"tokens":output_len}

    async def run(self, n_requests=100, concurrency=10, prompt_len=256, output_len=100):
        sem = asyncio.Semaphore(concurrency)
        t0 = time.perf_counter()
        results = await asyncio.gather(*[self.send_request(prompt_len,output_len,sem) for _ in range(n_requests)])
        elapsed = time.perf_counter()-t0
        ttfts = [r["ttft_ms"] for r in results]
        tpots = [r["tpot_ms"] for r in results]
        return {
            "n_requests":n_requests,"concurrency":concurrency,
            "ttft_p50":np.percentile(ttfts,50),"ttft_p99":np.percentile(ttfts,99),
            "tpot_p50":np.percentile(tpots,50),
            "throughput_rps":n_requests/elapsed,
            "throughput_tps":sum(r["tokens"] for r in results)/elapsed
        }

harness = InferenceBenchmarkHarness()
print("Benchmark results:")
for conc in [1, 10, 50]:
    r = asyncio.run(harness.run(n_requests=100, concurrency=conc))
    print(f"  concurrency={conc:>3}: TTFT_P99={r['ttft_p99']:.0f}ms throughput={r['throughput_tps']:.0f}tok/s")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- A benchmark harness runs standardized workloads against any OpenAI-compatible server, producing reproducible TTFT/TPOT/throughput reports..
- A benchmark harness runs standardized workloads against any OpenAI-compatible se....
- Day 75 implementation complete.
